# MOT20 Pedestrian Tracking Benchmark
## Part 1: Initial Setup & Helper Functions
First, we import our required libraries and define the helper functions needed to scan the Kaggle environment for the MOT20 dataset.

In [ ]:
!pip install -q ultralytics

In [ ]:
!pip install -q motmetrics ultralytics pandas pyyaml

In [ ]:
from ultralytics import YOLO
import os
from pathlib import Path
from collections import Counter
import pandas as pd

SEARCH_ROOTS = ["/kaggle/input", "/kaggle/working"]

def find_roots():
    candidates = []
    for root in SEARCH_ROOTS:
        if not os.path.exists(root): continue
        for current, dirs, files in os.walk(root):
            if (
                os.path.isdir(os.path.join(current, "train"))
                or os.path.isdir(os.path.join(current, "test"))
                or "mot20" in os.path.basename(current).lower()
            ):
                candidates.append(current)
    return sorted(set(candidates))

def find_sequences(split_dir):
    if not os.path.isdir(split_dir): return []
    return sorted({
        current for current, dirs, files in os.walk(split_dir)
        if os.path.isdir(os.path.join(current, "img1"))
    })

def count_images(img_dir):
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sum(Path(f).suffix.lower() in extensions for f in os.listdir(img_dir))

def read_seqinfo(seq_dir):
    path = os.path.join(seq_dir, "seqinfo.ini")
    info = {}
    if os.path.isfile(path):
        with open(path, encoding="utf-8", errors="ignore") as f:
            for line in f:
                if "=" in line:
                    key, value = line.strip().split("=", 1)
                    info[key.strip()] = value.strip()
    return info

## Part 2: Dataset Discovery
Now we will use our helper functions to locate the exact path of the MOT20 dataset and split it into training and testing sequences.

In [ ]:
print("MOT20 DATASET DISCOVERY")
candidates = find_roots()
print(f"Candidate directories: {len(candidates)}")

mot_root = None
for candidate in candidates:
    train = os.path.join(candidate, "train")
    if os.path.isdir(train) and find_sequences(train):
        mot_root = candidate
        break

if mot_root is None:
    raise RuntimeError("MOT20 dataset could not be detected.")

train_dir = os.path.join(mot_root, "train")
test_dir = os.path.join(mot_root, "test")
train_sequences = find_sequences(train_dir)
test_sequences = find_sequences(test_dir)

print(f"MOT root: {mot_root}")
print(f"Train sequences found: {len(train_sequences)}")

## Part 3: Ground Truth Validation
We need to ensure the dataset isn't corrupted. This cell reads the `gt.txt` files for all training sequences, counts the pedestrian bounding boxes, and checks for malformed rows.

In [ ]:
rows = []
for seq_dir in train_sequences:
    seq = os.path.basename(seq_dir)
    img_dir = os.path.join(seq_dir, "img1")
    gt_path = os.path.join(seq_dir, "gt", "gt.txt")

    has_img = os.path.isdir(img_dir)
    has_gt = os.path.isfile(gt_path)

    images = count_images(img_dir) if has_img else 0
    info = read_seqinfo(seq_dir)
    expected = info.get("seqLength")

    gt_rows, people, malformed = 0, 0, 0

    if has_gt:
        try:
            gt = pd.read_csv(gt_path, header=None, sep=",", dtype=str, engine="python")
            gt_rows = len(gt)
            if gt.shape[1] >= 8:
                numeric = gt.iloc[:, :8].apply(pd.to_numeric, errors="coerce")
                frame, obj_id, x, y, w, h, flag, cls = [numeric.iloc[:, i] for i in range(8)]
                
                people = int(((cls == 1) & (flag != 0)).sum())
                malformed = int((frame.isna() | x.isna() | y.isna() | w.isna() | h.isna() | flag.isna() | cls.isna()).sum())
        except Exception as e:
            print(f"GT error - {seq}: {e}")

    rows.append({
        "Sequence": seq, "Images": images, "Expected": expected, 
        "GT Rows": gt_rows, "Person GT": people, "Malformed": malformed
    })

df = pd.DataFrame(rows)
print("\nDATASET SUMMARY")
print(df.to_string(index=False))
if df["Malformed"].sum() > 0: print(" WARNING: Malformed Ground Truth data found!")
else: print("\n Dataset structure is valid and ready.")

## Part 4: YOLO Settings & Inference Config
Let's configure our environment variables, set the working directories for the outputs, and define the YOLO models we want to test.

In [ ]:
# === Paths ===
MOT_ROOT = Path(mot_root)
MOT20_TRAIN_DIR = MOT_ROOT / "train"

# Dynamically resolve output directory for cross-environment portability
OUTPUT_DIR = Path.cwd()
WORK_DIR = OUTPUT_DIR / "mot20_yolo26_benchmark"
VIDEO_FPS = 25.0

# YOLO Models 
DETECTOR_MODELS = ["yolo26s.pt", "yolo26m.pt"]

# Inference Hyperparameters (Tuned for MOT20 Density) 
DETECTOR_INFER = {
    "imgsz": 1280,       
    "conf": 0.1,         
    "iou": 0.7,          
    "max_det": 1000,     
    "classes": [0],      
    "device": None,      
}

print(f"Output directory set to: {WORK_DIR}")
print(f"Models to evaluate: {DETECTOR_MODELS}")

## Part 5: BoT-SORT Tracker Configuration
To track efficiently in dense crowds, we must tune BoT-SORT. We create a dictionary of presets, select the dense preset, and export it as a `.yaml` file for YOLO to use.

### ⚙️ BoT-SORT Dense Crowd Configuration

MOT20 contains extremely dense crowds. We use the following hyperparameter strategy to prevent the tracker from losing people when they are temporarily blocked by others:

| Hyperparameter | Value | Role in the Tracker |
| :--- | :--- | :--- |
| `track_high_thresh` | **0.30** | Minimum YOLO confidence to accept a bounding box. Lowered to catch partially hidden people. |
| `track_low_thresh` | **0.05** | Minimum confidence for the "second pass". Saves terrible predictions just in case they match a lost ID. |
| `track_buffer` | **45** | **Memory:** The number of frames (approx. 1.8 seconds) the AI will remember a lost person before deleting their ID. |
| `proximity_thresh` | **0.50** | **Spatial Mask:** Forces the AI to only check shirt colors if the box is physically close to the predicted location. |
| `gmc_method` | **sparseOptFlow** | **Camera Stabilization:** Cancels out shaky camera movement by tracking the background pixels. |

In [ ]:
import copy, yaml

BOTSORT_PRESETS = {
    "mot20_dense": {
        "tracker_type": "botsort",
        "track_high_thresh": 0.3,
        "track_low_thresh": 0.05,
        "new_track_thresh": 0.3,
        "track_buffer": 45,
        "match_thresh": 0.8,
        "fuse_score": True,
        "gmc_method": "sparseOptFlow", 
        "with_reid": True,
        "model": "auto", 
        "proximity_thresh": 0.5,
        "appearance_thresh": 0.25,
    }
}

ACTIVE_PRESET = "mot20_dense"
BOTSORT_CFG = copy.deepcopy(BOTSORT_PRESETS[ACTIVE_PRESET])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CUSTOM_TRACKER_PATH = OUTPUT_DIR / "custom_botsort.yaml"

with open(CUSTOM_TRACKER_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(dict(BOTSORT_CFG), f, sort_keys=False)

print(f" Saved Tracker Config '{ACTIVE_PRESET}' to: {CUSTOM_TRACKER_PATH}")

## Part 6: Prepare YOLO Dataset (Folders & Setup)
YOLO requires data in a specific structure. This cell builds the directory tree and checks if we have already extracted the dataset in a previous run to save time.

In [ ]:
import glob, shutil, cv2

DATASET_DIR = os.path.join(WORK_DIR, "dataset")
IMAGE_DIR = os.path.join(DATASET_DIR, "images")
LABEL_DIR = os.path.join(DATASET_DIR, "labels")
RESULT_DIR = os.path.join(WORK_DIR, "results")
DATA_YAML = os.path.join(DATASET_DIR, "mot20.yaml")
CSV_PATH = os.path.join(RESULT_DIR, "mot20_yolo26_comparison.csv")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

existing_images = glob.glob(os.path.join(IMAGE_DIR, "*"))
existing_labels = glob.glob(os.path.join(LABEL_DIR, "*.txt"))

NEEDS_EXTRACTION = True
if len(existing_images) > 0 and len(existing_images) == len(existing_labels):
    print(" YOLO dataset already prepared. Skipping generation.")
    NEEDS_EXTRACTION = False
else:
    print("Cleaning up old files before extraction...")
    for path in existing_images + existing_labels:
        try: os.remove(path)
        except: pass

## Part 7: Prepare YOLO Dataset (Bounding Box Conversion)
If the dataset isn't ready, this cell converts MOT20 absolute bounding boxes `[x, y, w, h]` into YOLO normalized bounding boxes `[class, x_center, y_center, width, height]` and links the images.

In [ ]:
if NEEDS_EXTRACTION:
    print("Extracting frames and generating YOLO labels... (This may take a moment)")
    total_images, total_boxes = 0, 0
    
    TRAIN_DIR = os.path.join(MOT_ROOT, "train")
    SEQUENCES = sorted([os.path.join(TRAIN_DIR, n) for n in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, n, "img1"))])

    for seq_dir in SEQUENCES:
        seq_name = os.path.basename(seq_dir)
        img_dir = os.path.join(seq_dir, "img1")
        gt_file = os.path.join(seq_dir, "gt", "gt.txt")
        
        # Get frame dimensions
        first_img = sorted(glob.glob(os.path.join(img_dir, "*")))[0]
        height, width = cv2.imread(first_img).shape[:2]

        gt_by_frame = {}
        with open(gt_file, "r", errors="ignore") as f:
            for line in f:
                parts = line.strip().split(",")
                if len(parts) < 8: continue
                try: frame_id, _, x, y, w, h, flag, class_id = map(float, parts[:8])
                except: continue

                if int(class_id) != 1 or int(flag) != 1 or w <= 0 or h <= 0: continue

                x, y = max(0, x), max(0, y)
                w, h = min(w, width - x), min(h, height - y)
                if w <= 0 or h <= 0: continue

                xc, yc = (x + w / 2) / width, (y + h / 2) / height
                gt_by_frame.setdefault(int(frame_id), []).append((0, xc, yc, w/width, h/height))
                total_boxes += 1

        for img_path in sorted(glob.glob(os.path.join(img_dir, "*.*"))):
            try: frame_id = int(Path(img_path).stem)
            except: continue

            out_name = f"{seq_name}_{Path(img_path).stem}"
            dest_img = os.path.join(IMAGE_DIR, out_name + Path(img_path).suffix.lower())
            
            try: os.symlink(img_path, dest_img)
            except: shutil.copy2(img_path, dest_img)

            with open(os.path.join(LABEL_DIR, out_name + ".txt"), "w") as f:
                for cls, xc, yc, nw, nh in gt_by_frame.get(frame_id, []):
                    f.write(f"{cls} {xc:.8f} {yc:.8f} {nw:.8f} {nh:.8f}\n")
            total_images += 1
            
    print(f"\n Converted {total_images:,} images & {total_boxes:,} boxes.")

# Generate YAML config for YOLO
with open(DATA_YAML, "w") as f:
    yaml.safe_dump({"path": DATASET_DIR, "train": "images", "val": "images", "nc": 1, "names": {0: "person"}}, f, sort_keys=False)

## Part 8: Detector Profiling Tools
Before evaluating, we define memory cleanup and benchmarking functions to calculate GFLOPs, Parameters, and Inference Latency without tracking overhead.

In [ ]:
import gc, time, torch

def reset_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def gpu_allocated():
    return torch.cuda.memory_allocated(0) / (1024 ** 3) if torch.cuda.is_available() else 0.0

def measure_gpu_speed(model, batch_size=8, img_size=1280):
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    img_paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.*")))[:batch_size]
    if not torch.cuda.is_available() or not img_paths: return None, None
    
    tensors = []
    for path in img_paths:
        img = cv2.resize(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB), (img_size, img_size))
        tensors.append(torch.from_numpy(img).permute(2, 0, 1).contiguous().float() / 255.0)
    
    batch = torch.stack(tensors).to(DEVICE)
    model.model.eval()
    
    with torch.inference_mode():
        for _ in range(10): model.model(batch) # Warmup
        torch.cuda.synchronize()
        times = []
        for _ in range(50):
            start = time.perf_counter()
            model.model(batch)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - start) * 1000 / batch_size)
            
    stable = sorted(times)[int(len(times)*0.1) : -int(len(times)*0.1)]
    latency = sum(stable) / len(stable)
    return latency, 1000 / latency

## Part 9: YOLO Detection Benchmark
Runs the detection validation (`mAP50`, `Precision`, `Recall`) using the dataset we created. Results are saved to a CSV file.

In [ ]:
results = []
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

for model_name in DETECTOR_MODELS:
    reset_gpu()
    print(f"\nRunning Validator on {model_name}...")
    row = {"Model": model_name, "Status": "FAILED", "Base_GPU_GB": gpu_allocated()}
    
    try:
        model = YOLO(model_name)
        model.to(DEVICE)
        
        row["Model_GPU_GB"] = gpu_allocated()
        row["Parameters_M"] = sum(p.numel() for p in model.model.parameters()) / 1e6
        
        latency, fps = measure_gpu_speed(model, img_size=DETECTOR_INFER["imgsz"])
        row["GPU_Inference_ms"], row["GPU_FPS"] = latency, fps
        
        metrics = model.val(
            data=DATA_YAML, imgsz=DETECTOR_INFER["imgsz"], device=DEVICE,
            max_det=DETECTOR_INFER["max_det"], conf=DETECTOR_INFER["conf"], 
            iou=DETECTOR_INFER["iou"], classes=[0], plots=False, verbose=False
        )
        
        row.update({
            "Precision": float(metrics.box.mp), "Recall": float(metrics.box.mr),
            "mAP50": float(metrics.box.map50), "mAP50-95": float(metrics.box.map),
            "Peak_GPU_Allocated_GB": torch.cuda.max_memory_allocated(0) / (1024**3) if torch.cuda.is_available() else None,
            "Status": "SUCCESS"
        })
        print(f" {model_name} Validation Complete.")
        
    except Exception as e: print(f" {model_name} failed: {e}")
    finally:
        results.append(row)
        del model
        reset_gpu()

results_df = pd.DataFrame(results)
results_df.to_csv(CSV_PATH, index=False)
display(results_df.style.format(precision=3))

## Part 10: Multi-Object Tracking Setup
Now we install TrackEval and set up the official HOTA metric evaluator to grade BoT-SORT's ability to maintain IDs over time.

### 📊 Evaluation Metrics Explained

When evaluating object tracking, detection accuracy and ID consistency are graded separately. Here is how to read the final benchmark scores:

| Metric | Full Name | What it measures | Goal |
| :--- | :--- | :--- | :--- |
| **mAP50** | Mean Average Precision | **Detection:** How well YOLO finds people, regardless of tracking. | Higher ↑ |
| **MOTA** | Multi-Object Tracking Accuracy | **Overall:** Combines missed people, false ghosts, and ID swaps into one score. | Higher ↑ |
| **IDF1** | ID F1-Score | **Consistency:** How well the tracker keeps the *exact same ID* on a person without swapping. | Higher ↑ |
| **IDSW** | ID Switches | **Errors:** The total number of times the tracker accidentally gave an old person a new ID number. | Lower ↓ |
| **HOTA** | Higher Order Tracking Accuracy | **The Gold Standard:** A modern metric that perfectly balances detection accuracy and ID consistency. | Higher ↑ |

In [ ]:
import subprocess, sys

# Install tracking metric libraries
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "motmetrics", "lap", "git+https://github.com/JonathonLuiten/TrackEval.git"], check=True)

import numpy as np

# === NUMPY COMPATIBILITY FIX ===
if not hasattr(np, "asfarray"): np.asfarray = lambda a, *args, **kwargs: np.asarray(a, dtype=float)
if not hasattr(np, "float"): np.float = float
if not hasattr(np, "int"): np.int = int
if not hasattr(np, "bool"): np.bool = bool
# ===============================

import trackeval

def run_trackeval_hota(gt_folder, trackers_folder, tracker_name, seq_names):
    """Official HOTA / CLEAR / Identity via TrackEval."""
    eval_config = trackeval.Evaluator.get_default_eval_config()
    eval_config.update({
        "DISPLAY_LESS_PROGRESS": True, 
        "PRINT_RESULTS": False, 
        "USE_PARALLEL": False
    })
    
    dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    dataset_config.update({
        "GT_FOLDER": str(gt_folder), 
        "TRACKERS_FOLDER": str(trackers_folder),
        "OUTPUT_FOLDER": str(Path(trackers_folder) / "_trackeval_out"),
        "TRACKERS_TO_EVAL": [tracker_name], 
        "CLASSES_TO_EVAL": ["pedestrian"],
        "BENCHMARK": "MOT20", 
        "SPLIT_TO_EVAL": "train", 
        "SKIP_SPLIT_FOL": True,
        "DO_PREPROC": True, 
        "PRINT_CONFIG": False, 
        "TRACKER_SUB_FOLDER": "data",
        "SEQ_INFO": {s: None for s in seq_names},
    })
    
    metrics = [trackeval.metrics.HOTA({"METRICS": ["HOTA"]}), 
               trackeval.metrics.CLEAR({"METRICS": ["CLEAR"]}), 
               trackeval.metrics.Identity({"METRICS": ["Identity"]})]
    
    evaluator = trackeval.Evaluator(eval_config)
    dataset = trackeval.datasets.MotChallenge2DBox(dataset_config)
    out, _ = evaluator.evaluate([dataset], metrics)
    
    ped = out[list(out.keys())[0]][tracker_name]["COMBINED_SEQ"]["pedestrian"]
    
    def _mean(x):
        x = np.asarray(x, dtype=float)
        return float(np.mean(x))

    return {
        "HOTA (%)": round(_mean(ped["HOTA"]["HOTA"]) * 100, 1),
        "DetA (%)": round(_mean(ped["HOTA"]["DetA"]) * 100, 1),
        "AssA (%)": round(_mean(ped["HOTA"]["AssA"]) * 100, 1),
        "MOTA (%)": round(float(ped["CLEAR"]["MOTA"]) * 100, 1),
        "IDF1 (%)": round(float(ped["Identity"]["IDF1"]) * 100, 1),
        "ID Switches": int(ped["CLEAR"]["IDSW"]),
        "Frag": int(ped["CLEAR"]["Frag"]),
    }

## Part 11: Tracking Execution
We loop through each model, run YOLO + BoT-SORT on every sequence to generate tracking text files, and pass them to TrackEval for scoring.

In [ ]:
tracking_results = []
TRAIN_DIR_PATH = MOT_ROOT / "train"
SEQUENCES_NAMES = [Path(s).name for s in sorted(glob.glob(os.path.join(str(TRAIN_DIR_PATH), "*")))]

for model_name in DETECTOR_MODELS:
    tracker_name = model_name.split(".")[0]
    print(f"\nTracking with {tracker_name} + BoT-SORT...")
    
    model = YOLO(model_name)
    model_output_dir = OUTPUT_DIR / tracker_name / "data"
    model_output_dir.mkdir(parents=True, exist_ok=True)
    
    for seq in SEQUENCES_NAMES:
        seq_dir = TRAIN_DIR_PATH / seq / "img1"
        img_paths = sorted(list(seq_dir.glob("*.jpg")) + list(seq_dir.glob("*.png")))
        if not img_paths: continue
            
        # Accumulate tracking data in memory to eliminate I/O bottleneck
        seq_results = []
        
        for idx, img_path in enumerate(img_paths):
            res = model.track(
                source=str(img_path), tracker=str(CUSTOM_TRACKER_PATH), persist=(idx != 0),
                device=DEVICE, imgsz=DETECTOR_INFER["imgsz"], conf=DETECTOR_INFER["conf"], 
                iou=DETECTOR_INFER["iou"], classes=[0], verbose=False
            )
            for r in res:
                if r.boxes.id is None: continue
                for box, tid, conf in zip(r.boxes.xywh, r.boxes.id.int().tolist(), r.boxes.conf.tolist()):
                    x, y, w, h = box.tolist()
                    seq_results.append(f"{int(img_path.stem)},{tid},{x-w/2:.2f},{y-h/2:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1\n")
        
        # Perform a single batch write for the entire sequence
        with open(model_output_dir / f"{seq}.txt", "w", encoding="utf-8") as f:
            f.writelines(seq_results)
            
    try:
        metrics = run_trackeval_hota(TRAIN_DIR_PATH, OUTPUT_DIR, tracker_name, SEQUENCES_NAMES)
        metrics["Model"] = tracker_name
        tracking_results.append(metrics)
        print(f" HOTA Evaluation Complete for {tracker_name}")
    except Exception as e: print(f" TrackEval failed: {e}")

if tracking_results:
    display(pd.DataFrame(tracking_results).set_index("Model"))

## Part 12: Visual Error Analysis Setup

To debug the tracker, we need to compare its output with the Ground Truth and evaluate tracking identity errors across frames. This function takes prediction files, uses the Hungarian algorithm to track IDs, and draws Green (True Positive), Yellow (ID Switch), Blue (False Positive), and Red (False Negative) boxes.

| Box Color | Type | Meaning | What it tells us to fix |
| :--- | :--- | :--- | :--- |
| 🟩 **Green** | **True Positive (TP)** | The AI successfully tracked a real person with the correct ID. | *(Working perfectly)* |
| 🟨 **Yellow**| **ID Switch (IDSW)** | The AI tracked the person but assigned them a new/different ID. | Tune tracker memory or association thresholds. |
| 🟦 **Blue** | **False Positive (FP)** | The AI drew a box, but nobody is actually there. | Raise YOLO's confidence threshold. |
| 🟥 **Red** | **False Negative (FN)** | A real person is there, but the AI missed them. | Lower YOLO's confidence threshold. |

In [ ]:
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

# --- Helper Function: Bounding Box IoU ---
def bbox_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[0] + boxA[2], boxB[0] + boxB[2]), min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = boxA[2] * boxA[3]
    boxBArea = boxB[2] * boxB[3]
    return interArea / float(boxAArea + boxBArea - interArea + 1e-5)

# --- Configuration & Paths ---
SEQ_NAME = 'MOT20-01'
MOT20_IMG_DIR = TRAIN_DIR_PATH / SEQ_NAME / 'img1'
GT_FILE = TRAIN_DIR_PATH / SEQ_NAME / 'gt/gt.txt'
PRED_FILE = OUTPUT_DIR / f'yolo26s/data/{SEQ_NAME}.txt'

FRAME_ID = 150  # Selected frame for visual analysis
OUTPUT_PATH = OUTPUT_DIR / 'before_after_sample.jpg'
IOU_THRESH = 0.5

## Part 13: Generate Single Frame Analysis

We process tracking history up to frame `#150`, run one-to-one matching using the Hungarian algorithm based on Intersection over Union (IoU) cost, detect identity switches across frames, draw the color-coded bounding boxes with ID labels, and stitch a "Before and After" comparison image.

In [ ]:
# --- Load DataFrames ---
gt_df = pd.read_csv(GT_FILE, header=None, names=['frame', 'id', 'x', 'y', 'w', 'h', 'mark', 'cls', 'vis'])
gt_df = gt_df[(gt_df['mark'] == 1) & (gt_df['cls'] == 1)]

try:
    pred_df = pd.read_csv(PRED_FILE, header=None, names=['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'x3d', 'y3d', 'z3d'])
except FileNotFoundError:
    print(" Prediction file not found. Did the tracker run successfully?")
    pred_df = None

img_path = MOT20_IMG_DIR / f"{FRAME_ID:06d}.jpg"

if img_path.exists() and pred_df is not None:
    before_img = cv2.imread(str(img_path))
    after_img = before_img.copy()

    # Track GT ID -> Pred ID associations across frames up to FRAME_ID
    gt_to_pred_history = {}
    
    frame_tp, frame_fp, frame_fn, frame_idsw = 0, 0, 0, 0
    draw_commands = []

    # Process all frames up to FRAME_ID to track identity continuity
    for f in range(1, FRAME_ID + 1):
        gt_f = gt_df[gt_df['frame'] == f]
        pred_f = pred_df[pred_df['frame'] == f]

        gt_boxes = gt_f[['x', 'y', 'w', 'h']].values
        gt_ids = gt_f['id'].values
        pred_boxes = pred_f[['x', 'y', 'w', 'h']].values
        pred_ids = pred_f['id'].values

        if len(gt_boxes) == 0 or len(pred_boxes) == 0:
            continue

        # Build IoU Cost Matrix (Cost = 1 - IoU)
        iou_matrix = np.zeros((len(gt_boxes), len(pred_boxes)))
        for i, g_box in enumerate(gt_boxes):
            for j, p_box in enumerate(pred_boxes):
                iou_matrix[i, j] = bbox_iou(g_box, p_box)

        cost_matrix = 1.0 - iou_matrix
        row_ind, col_ind = linear_sum_assignment(cost_matrix)

        matched_gt_indices = set()
        matched_pred_indices = set()

        for r, c in zip(row_ind, col_ind):
            if iou_matrix[r, c] >= IOU_THRESH:
                matched_gt_indices.add(r)
                matched_pred_indices.add(c)

                g_id = gt_ids[r]
                p_id = pred_ids[c]

                # Check for ID switch candidate
                is_idsw = False
                if g_id in gt_to_pred_history and gt_to_pred_history[g_id] != p_id:
                    is_idsw = True

                gt_to_pred_history[g_id] = p_id

                if f == FRAME_ID:
                    if is_idsw:
                        frame_idsw += 1
                        color = (0, 255, 255)  # Yellow: ID Switch candidate
                    else:
                        frame_tp += 1
                        color = (0, 255, 0)    # Green: TP
                    
                    label = f"GT:{g_id}|Trk:{p_id}"
                    draw_commands.append((pred_boxes[c], color, label))

        if f == FRAME_ID:
            # Unmatched GT -> False Negatives (Red)
            for i, g_box in enumerate(gt_boxes):
                if i not in matched_gt_indices:
                    frame_fn += 1
                    draw_commands.append((g_box, (0, 0, 255), f"GT:{gt_ids[i]}|Trk:None"))

            # Unmatched Predictions -> False Positives (Blue)
            for j, p_box in enumerate(pred_boxes):
                if j not in matched_pred_indices:
                    frame_fp += 1
                    draw_commands.append((p_box, (255, 0, 0), f"GT:None|Trk:{pred_ids[j]}"))

    # --- Bounding Box & Label Overlay ---
    for box, color, label in draw_commands:
        x, y, w, h = map(int, box)
        cv2.rectangle(after_img, (x, y), (x + w, y + h), color, 2)
        
        # Background rectangle for text contrast
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
        cv2.rectangle(after_img, (x, max(0, y - 15)), (x + text_w, max(15, y)), color, -1)
        cv2.putText(after_img, label, (x, max(12, y - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1, cv2.LINE_AA)

    # --- Header Banners & Side-by-Side Canvas ---
    banner_height = 50
    cv2.rectangle(before_img, (0, 0), (before_img.shape[1], banner_height), (0, 0, 0), -1)
    cv2.putText(before_img, f"BEFORE: Frame {FRAME_ID}", (15, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.rectangle(after_img, (0, 0), (after_img.shape[1], banner_height), (0, 0, 0), -1)
    banner_text = f"AFTER: TP(Grn):{frame_tp} | IDSW(Yel):{frame_idsw} | FP(Blu):{frame_fp} | FN(Red):{frame_fn}"
    cv2.putText(after_img, banner_text, (15, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    combined_img = np.hstack([before_img, after_img])
    cv2.imwrite(str(OUTPUT_PATH), combined_img)

    # Console Summary
    print(f"=== Visual Analysis Summary for Frame {FRAME_ID} ===")
    print(f" True Positives (TP - Green): {frame_tp}")
    print(f" ID-Switch Candidates (IDSW - Yellow): {frame_idsw}")
    print(f" False Positives (FP - Blue): {frame_fp}")
    print(f" False Negatives (FN - Red): {frame_fn}")
    print(f" Visual image saved to: {OUTPUT_PATH}")
else:
    print(f" Error: Could not locate image at {img_path} or predictions were missing.")

## Part 14: Display Visual Results
Finally, we load the side-by-side image and render it directly in the Jupyter Notebook using Matplotlib.

In [ ]:
# --- Render Side-by-Side Comparison Plot ---
if OUTPUT_PATH.exists() and 'combined_img' in locals():
    plt.figure(figsize=(22, 11))
    plt.imshow(cv2.cvtColor(combined_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Image output not available for display.")